#Taller 6 de física computacional
Nombre del taller: Cálculo de π y convergencia

Estudiante: Cristian Fernando Castro Torres

#Parte B

#B.1

```
MÓDULO del modelo
    constantes: a_V, a_S, a_C, a_A, M_n, M_H
    funciones de los cuatro términos               (entradas: Z y/o A; salida: el término [MeV])
    función E_B(Z, A)                              (usa las cuatro funciones anteriores)
    función M(Z, A)                                (usa E_B)
    procedimiento Z_estable                        (entrada: A; resultados: el Z de máximo enlace y su E_B)
        [COMPLETE 1: Inicializar EB_minimo con el valor de E_B(1, A) y Z_optimo en 1.
                     Iniciar un ciclo para Z desde 2 hasta A.
                     En cada iteración, calcular E_B(Z, A).
                     Si E_B(Z, A) < EB_minimo, actualizar EB_minimo = E_B(Z, A) y Z_optimo = Z.]
        [JUSTIFIQUE: Según la regla de diseño del curso (sección 3.3), una función (function) solo debe devolver un único resultado escalar a través de su nombre. Dado que aquí necesitamos devolver dos resultados distintos (el valor de Z y la energía E_B correspondiente), es obligatorio usar una subrutina (subroutine) donde estos valores se entreguen mediante argumentos declarados con intent(out).]


PROGRAMA PRINCIPAL (usa el módulo)
    para cada uno de los 8 núcleos, en el orden del enunciado:
        escribir una fila de nucleos.dat: Z, A, los cuatro términos, E_B y M
    para A = 4, 5, ..., 260:
        obtener el Z de máximo enlace y su E_B con Z_estable
        escribir una fila de valle.dat: A, Z, E_B
        [COMPLETE 2: Comparar el E_B devuelto por Z_estable con EB_global.
                     Si es menor (más negativo, por tanto más ligado),
                     actualizar EB_global = E_B, Z_global = Z, y A_global = A.]
    escribir el núcleo más ligado en maximo.dat
```

#B.3

In [50]:
%%bash
mkdir -p taller_nucleo_atomico
cat << 'EOF' > taller_nucleo_atomico/modulo_nucleo.f90
! Taller 6 -- Módulo de la fórmula semiempírica
module modulo_nucleo
  implicit none

  ! Constantes del modelo
  integer, parameter :: dp = kind(1.0d0)
  real(dp), parameter :: a_V = 15.8_dp
  real(dp), parameter :: a_S = 18.3_dp
  real(dp), parameter :: a_C = 0.71_dp
  real(dp), parameter :: a_A = 92.7_dp
  real(dp), parameter :: M_n = 939.57_dp
  real(dp), parameter :: M_H = 938.78_dp

contains

  ! Término de Volumen
  real(dp) function e_vol(Z, A)
    integer, intent(in) :: Z, A
    e_vol = -a_V
  end function e_vol

  ! Término de Superficie
  real(dp) function e_sup(Z, A)
    integer, intent(in) :: Z, A
    e_sup = a_S * (real(A, dp)**(-1.0_dp/3.0_dp))
  end function e_sup

  ! Término de Coulomb
  real(dp) function e_coul(Z, A)
    integer, intent(in) :: Z, A
    e_coul = a_C * (real(Z, dp)**2) * (real(A, dp)**(-4.0_dp/3.0_dp))
  end function e_coul

  ! Término de Asimetría
  real(dp) function e_asim(Z, A)
    integer, intent(in) :: Z, A
    e_asim = a_A * (real(Z, dp) - real(A, dp)/2.0_dp)**2 / (real(A, dp)**2)
  end function e_asim

  ! Energía de enlace por nucleón total (E_B)
  real(dp) function E_B(Z, A)
    integer, intent(in) :: Z, A
    E_B = e_vol(Z, A) + e_sup(Z, A) + e_coul(Z, A) + e_asim(Z, A)
  end function E_B

  ! Masa atómica aproximada (M)
  real(dp) function masa(Z, A)
    integer, intent(in) :: Z, A
    masa = real(Z, dp)*M_H + real(A - Z, dp)*M_n + real(A, dp)*E_B(Z, A)
  end function masa

  ! Búsqueda del isótopo más estable para un A dado
  subroutine Z_estable(A, Z_min, EB_min)
    integer, intent(in) :: A
    integer, intent(out) :: Z_min
    real(dp), intent(out) :: EB_min

    integer :: Z
    real(dp) :: eb_actual

    ! Inicialización usando Z=1 como referencia base
    EB_min = E_B(1, A)
    Z_min = 1

    ! Búsqueda del menor E_B (mayor estabilidad)
    do Z = 2, A
       eb_actual = E_B(Z, A)
       if (eb_actual < EB_min) then
          EB_min = eb_actual
          Z_min = Z
       end if
    end do
  end subroutine Z_estable

end module modulo_nucleo
EOF


In [51]:
%%writefile taller_nucleo_atomico/programa_nucleo.f90
! Taller 6 -- Programa principal
program programa_nucleo
  use modulo_nucleo
  implicit none

  integer :: u_nuc, u_val, u_max, i, A, Z
  integer :: Z_valle, A_global, Z_global
  real(dp) :: EB_valle, EB_global

  ! Datos para la primera pregunta: 244Pu, 70Ge, 179Hf, 38Ar, 56Fe, 40Ca, 235U, 238U
  integer, dimension(8) :: arr_Z = [94, 32, 72, 18, 26, 20, 92, 92]
  integer, dimension(8) :: arr_A = [244, 70, 179, 38, 56, 40, 235, 238]

  ! 1. Generar nucleos.dat
  open(newunit=u_nuc, file='nucleos.dat', status='replace')
  do i = 1, 8
     Z = arr_Z(i)
     A = arr_A(i)
     ! 8 columnas: Z, A, E_vol, E_sup, E_coul, E_asim, E_B, M
     write(u_nuc, '(2I5, 5F15.8, F15.4)') Z, A, e_vol(Z,A), e_sup(Z,A), e_coul(Z,A), e_asim(Z,A), E_B(Z,A), masa(Z,A)
  end do
  close(u_nuc)

  ! 2 y 3. Generar valle.dat y encontrar el núcleo globalmente más ligado
  open(newunit=u_val, file='valle.dat', status='replace')
  EB_global = 1.0e10_dp ! Valor inicial arbitrariamente alto para asegurar reemplazo

  do A = 4, 260
     call Z_estable(A, Z_valle, EB_valle)
     write(u_val, '(2I5, F15.8)') A, Z_valle, EB_valle

     ! Filtrar el mínimo global a lo largo de toda la iteración
     if (EB_valle < EB_global) then
        EB_global = EB_valle
        A_global = A
        Z_global = Z_valle
     end if
  end do
  close(u_val)

  ! Escribir maximo.dat con el resultado global
  open(newunit=u_max, file='maximo.dat', status='replace')
  write(u_max, '(2I5, F15.8)') A_global, Z_global, EB_global
  close(u_max)

end program programa_nucleo

Overwriting taller_nucleo_atomico/programa_nucleo.f90


#Parte c

In [52]:
%%writefile taller_nucleo_atomico/analisis_nucleo.py
"""Taller 6 -- Módulo de análisis en Python."""
import numpy as np
import matplotlib.pyplot as plt

def leer_valle(archivo):
    """
    Lee la tabla valle.dat generada por Fortran.
    """
    datos = np.loadtxt(archivo)
    return datos[:, 0], datos[:, 1], datos[:, 2]

def encontrar_maximo(A, Z, E_B):
    """
    Encuentra el núcleo más ligado (menor valor de E_B) usando un ciclo explícito.
    """
    EB_min = 1.0e10
    A_max = 0
    Z_max = 0

    for i in range(len(E_B)):
        if E_B[i] < EB_min:
            EB_min = E_B[i]
            A_max = A[i]
            Z_max = Z[i]

    return A_max, Z_max, EB_min

def escribir_maximo(archivo, A_max, Z_max, EB_min):
    """
    Escribe el núcleo más ligado en un archivo.
    """
    with open(archivo, 'w') as f:
        f.write(f"{int(A_max):5d}{int(Z_max):5d}{EB_min:15.8f}\n")

def graficar_curva(A, E_B, experimentales, archivo_salida):
    """
    Genera y guarda la gráfica de B/A = -E_B contra A, con datos experimentales.
    """
    B_A_modelo = -E_B

    plt.figure(figsize=(9, 5))
    plt.plot(A, B_A_modelo, '-', color="#162A54", label="Modelo (Curva de máximo enlace)")

    A_exp = [dato[2] for dato in experimentales]
    B_A_exp = [dato[3] for dato in experimentales]

    plt.plot(A_exp, B_A_exp, 'o', color="#C9972C", label="Datos experimentales (AME2020)")

    plt.xlabel("Número de nucleones (A)")
    plt.ylabel("Energía de enlace por nucleón B/A [MeV]")
    plt.title("Energía de Enlace por Nucleón vs A (Modelo de Gota Líquida)")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(archivo_salida)
    plt.close()

Overwriting taller_nucleo_atomico/analisis_nucleo.py


In [53]:
# 1. Compilar y ejecutar el código Fortran primero para generar los archivos .dat
!gfortran -c taller_nucleo_atomico/modulo_nucleo.f90 -o taller_nucleo_atomico/modulo_nucleo.o
!gfortran -c taller_nucleo_atomico/programa_nucleo.f90 -o taller_nucleo_atomico/programa_nucleo.o
!gfortran taller_nucleo_atomico/modulo_nucleo.o taller_nucleo_atomico/programa_nucleo.o -o taller_nucleo_atomico/programa_nucleo
!cd taller_nucleo_atomico && ./programa_nucleo

# C.4 -- Uso del módulo de análisis
import sys, importlib, os

# Datos experimentales: (símbolo, Z, A, B/A [MeV]) -- AME2020
EXPERIMENTALES = [
    ("He-4",    2,   4, 7.0739),
    ("C-12",    6,  12, 7.6801),
    ("O-16",    8,  16, 7.9762),
    ("Ca-40",  20,  40, 8.5513),
    ("Fe-56",  26,  56, 8.7903),
    ("Ni-62",  28,  62, 8.7945),
    ("Sn-120", 50, 120, 8.5045),
    ("Pb-208", 82, 208, 7.8675),
    ("U-235",  92, 235, 7.5909),
    ("U-238",  92, 238, 7.5701),
]

# Forzamos a Python a reconocer la carpeta exacta
ruta_absoluta = os.path.abspath("taller_nucleo_atomico")
if ruta_absoluta not in sys.path:
    sys.path.insert(0, ruta_absoluta)

import analisis_nucleo
a = importlib.reload(analisis_nucleo)

# 2. Llamadas a las funciones:
A_valle, Z_valle, EB_valle = analisis_nucleo.leer_valle("taller_nucleo_atomico/valle.dat")
A_max, Z_max, EB_min = analisis_nucleo.encontrar_maximo(A_valle, Z_valle, EB_valle)
analisis_nucleo.escribir_maximo("taller_nucleo_atomico/maximo_python.dat", A_max, Z_max, EB_min)
analisis_nucleo.graficar_curva(A_valle, EB_valle, EXPERIMENTALES, "taller_nucleo_atomico/curva_enlace.png")

print(f"Búsqueda finalizada. Núcleo más ligado: A={int(A_max)}, Z={int(Z_max)}, E_B={EB_min:.8f} MeV")

Búsqueda finalizada. Núcleo más ligado: A=65, Z=29, E_B=-8.69506906 MeV


#D. PASS/FAIL

In [54]:
# ===================== VERIFICACIÓN — NO MODIFICAR =====================
from pathlib import Path
import numpy as np

CARPETA = Path("taller_nucleo_atomico")
veredictos = []

def informar(nombre, condicion, detalle=""):
    veredictos.append(bool(condicion))
    print(f"{'PASS' if condicion else 'FAIL'} | {nombre}" + (f" — {detalle}" if detalle else ""))

def leer_salida(nombre, filas, columnas, origen="fortran"):
    ruta = CARPETA / nombre
    if not ruta.is_file():
        raise ValueError(f"falta {ruta}")
    if origen == "fortran":
        fuentes = [CARPETA / "modulo_nucleo.f90", CARPETA / "programa_nucleo.f90"]
    else:
        fuentes = [CARPETA / "analisis_nucleo.py", CARPETA / "valle.dat"]
    for fuente in fuentes:
        if not fuente.is_file():
            raise ValueError(f"falta {fuente}")
        if ruta.stat().st_mtime < fuente.stat().st_mtime:
            raise ValueError(f"{nombre} es anterior a {fuente.name}: vuelva a ejecutar el cálculo correspondiente")
    datos = np.loadtxt(ruta, ndmin=2)
    if datos.shape != (filas, columnas) or not np.isfinite(datos).all():
        raise ValueError(f"formato incorrecto en {nombre}: se esperaban {filas} filas y {columnas} columnas finitas")
    return datos

# Implementación de referencia exclusiva de la verificación.
def referencia_terminos(z, a):
    z, a = np.asarray(z, dtype=float), np.asarray(a, dtype=float)
    return np.broadcast_arrays(-15.8 + np.zeros_like(a), 18.3*a**(-1.0/3.0),
                               0.71*z**2*a**(-4.0/3.0), 92.7*(z-a/2.0)**2/a**2)

def referencia_enlace(z, a):
    return sum(referencia_terminos(z, a))

def leer_valle():
    datos = leer_salida("valle.dat", 257, 3)
    if not np.array_equal(datos[:, 0], np.arange(4, 261)):
        raise ValueError("valle.dat debe tener A = 4, 5, ..., 260, en ese orden")
    if not np.array_equal(datos[:, 1], np.rint(datos[:, 1])):
        raise ValueError("los valores de Z deben ser enteros")
    if np.any((datos[:, 1] < 1) | (datos[:, 1] > datos[:, 0])):
        raise ValueError("cada fila debe cumplir 1 <= Z <= A")
    return datos

# 1) Ocho núclidos: todos los términos, el enlace y la masa.
try:
    nuc = leer_salida("nucleos.dat", 8, 8)
    orden = np.array([(94,244), (32,70), (72,179), (18,38), (26,56), (20,40), (92,235), (92,238)])
    if not np.array_equal(nuc[:, :2], orden):
        raise ValueError("Z y A deben ser enteros y seguir el orden del enunciado")
    z, a = orden[:, 0], orden[:, 1]
    terminos = np.column_stack(referencia_terminos(z, a))
    eb = terminos.sum(axis=1)
    masa = z*938.78 + (a-z)*939.57 + a*eb
    ok = (np.allclose(nuc[:, 2:6], terminos, rtol=0, atol=1e-7)
          and np.allclose(nuc[:, 6], eb, rtol=0, atol=1e-7)
          and np.allclose(nuc[:, 7], masa, rtol=0, atol=1e-3)
          and np.allclose(nuc[:, 2:6].sum(axis=1), nuc[:, 6], rtol=0, atol=1e-7))
    informar("Caso 1: ocho núclidos", ok, f"Pu-244: E_B = {nuc[0,6]:.8f} MeV; M = {nuc[0,7]:.4f} MeV")
except Exception as error:
    informar("Caso 1: ocho núclidos", False, str(error))

# 2) La búsqueda obtiene el máximo enlace para cada A del intervalo.
try:
    valle = leer_valle()
    correctos = []
    for a, z, eb in valle:
        candidatos = np.arange(1, int(a)+1)
        energias = referencia_enlace(candidatos, a)
        correctos.append(abs(float(referencia_enlace(z, a)) - energias.min()) <= 1e-7)
    fila = valle[valle[:, 0] == 208][0]
    ok = all(correctos) and fila[1] == 82 and abs(fila[2] - (-7.800561449221129)) <= 1e-7
    informar("Caso 2: máximo enlace para cada A", ok, f"{sum(correctos)}/257 búsquedas correctas; A=208: Z={fila[1]:.0f}")
except Exception as error:
    informar("Caso 2: máximo enlace para cada A", False, str(error))

# 3) Energías de todas las filas y referencia de U-238.
try:
    valle = leer_valle()
    esperado = referencia_enlace(valle[:, 1], valle[:, 0])
    fila = valle[valle[:, 0] == 238][0]
    ok = (np.allclose(valle[:, 2], esperado, rtol=0, atol=1e-7)
          and fila[1] == 92 and abs(fila[2] - (-7.579586363043683)) <= 1e-7)
    informar("Caso 3: energías de la curva y U-238", ok, f"A=238: Z={fila[1]:.0f}; E_B={fila[2]:.8f} MeV")
except Exception as error:
    informar("Caso 3: energías de la curva y U-238", False, str(error))

# 4) Mínimo global: dos lenguajes y archivo de imagen.
try:
    valle = leer_valle()
    mf = leer_salida("maximo.dat", 1, 3)[0]
    mp = leer_salida("maximo_python.dat", 1, 3, origen="python")[0]
    referencia = valle[np.argmin(valle[:, 2])]
    def coincide(fila):
        return (np.array_equal(fila[:2], referencia[:2])
                and abs(fila[2]-referencia[2]) <= 1e-7)
    imagen = CARPETA / "curva_enlace.png"
    png_ok = (imagen.is_file() and imagen.stat().st_size > 8
              and imagen.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n")
    ok = (coincide(mf) and coincide(mp)
          and abs(referencia[2] - (-8.69506905766211)) <= 1e-7 and png_ok)
    informar("Caso 4: mínimo global e imagen", ok,
             f"Fortran: A={mf[0]:.0f}, Z={mf[1]:.0f}; Python: A={mp[0]:.0f}, Z={mp[1]:.0f}; PNG: {'presente' if png_ok else 'falta o formato incorrecto'}")
except Exception as error:
    informar("Caso 4: mínimo global e imagen", False, str(error))

print(f"\nResumen: {sum(veredictos)}/4 PASS")
print("La arquitectura, los requisitos y la interpretación se revisan por separado.")

PASS | Caso 1: ocho núclidos — Pu-244: E_B = -7.53611738 MeV; M = 227342.0074 MeV
PASS | Caso 2: máximo enlace para cada A — 257/257 búsquedas correctas; A=208: Z=82
PASS | Caso 3: energías de la curva y U-238 — A=238: Z=92; E_B=-7.57958636 MeV
PASS | Caso 4: mínimo global e imagen — Fortran: A=65, Z=29; Python: A=65, Z=29; PNG: presente

Resumen: 4/4 PASS
La arquitectura, los requisitos y la interpretación se revisan por separado.


#Parte E

1. Utilizando los datos del modelo, al comparar un núcleo liviano como el Calcio-40 frente a uno pesado como el Uranio-238, se observa que en el Calcio-40 el término de superficie penaliza el enlace con aproximadamente 5.35 MeV, dominando sobre su término de Coulomb de 2.08 MeV, mientras que en el Uranio-238 la situación se invierte y la repulsión de Coulomb penaliza fuertemente con 4.07 MeV frente a los 2.95 MeV de la superficie. Esta dinámica ocurre porque el efecto de superficie disminuye a medida que el núcleo crece geométricamente, pero simultáneamente la repulsión electrostática crece de forma cuadrática con la cantidad de protones. La competencia directa entre un término desestabilizador que decae y otro que aumenta rápidamente provoca que exista un punto de equilibrio óptimo en masas intermedias, generando el característico máximo en la curva de energía de enlace por nucleón.

2. La diferencia teórica de energía de enlace entre el máximo global predicho por el modelo para el $A=65$ y el valor evaluado en la curva teórica para el $A=62$ con $Z=28$ es de apenas $0.0053$ MeV, lo que representa una discrepancia minúscula de aproximadamente $0.061\%$ respecto a la magnitud en ese punto. Aunque numéricamente la diferencia es insignificante, el modelo falla en predecir empíricamente al Níquel-62 como el más estable porque la fórmula de la gota líquida es una aproximación macroscópica que asume al núcleo como un fluido continuo, omitiendo por completo los efectos de la estructura cuántica interna. Al ignorar el modelo cuántico de capas, la fórmula no puede contabilizar la enorme estabilidad adicional que otorgan los "números mágicos" de la naturaleza, como los 28 protones exactos que posee el isótopo del Níquel para cerrar una capa energética completa.
3. Al evaluar el modelo base de cuatro términos para $A=56$, el núcleo con mayor enlace por nucleón resulta ser el Manganeso ($Z=25$), superando al Hierro ($Z=26$) al tener una energía total teóricamente más ligada por una diferencia de $1.198$ MeV. Sin embargo, al introducir la corrección de energía total del quinto término de apareamiento con $a_P \approx 12$ MeV, el Manganeso sufre una penalización de $+1.604$ MeV por tener un número impar tanto de protones como de neutrones, mientras que el Hierro recibe una bonificación de $-1.604$ MeV al ser un núcleo par-par. Al calcular nuevamente sus energías totales ajustadas, el orden de estabilidad se invierte por completo por una diferencia final de $2.010$ MeV a favor del Hierro, demostrando que la preferencia cuántica de los nucleones por formar pares es el factor decisivo que lo hace físicamente más estable.
4. Calculando la energía liberada en la fisión simétrica idealizada del U-236 en dos fragmentos idénticos de Pd-118 mediante la diferencia de sus masas equivalentes teóricas, se obtiene un valor $Q$ de aproximadamente $180.3$ MeV. Este resultado teórico concuerda notablemente bien en orden de magnitud con la escala típica observable de unos 200 MeV por fisión del uranio. Sin embargo, este canal simétrico idealizado no reproduce una fisión real porque en la naturaleza la ruptura del núcleo es predominantemente asimétrica debido a la influencia de los niveles de energía de capas cuánticas, y además la resta directa de masas asume erróneamente que toda la energía se convierte en masa de los fragmentos, omitiendo la fracción térmica que se disipa de inmediato a través de la emisión de neutrones libres y rayos gamma prontos.   

1. Usando `nucleos.dat`, elija un núcleo relativamente liviano y otro pesado e identifíquelos. Compare los términos de superficie y Coulomb en ambos casos. ¿Cuál de esos dos términos penaliza más el enlace en cada uno? Explique cómo su competencia contribuye a que la curva $B/A$ presente un máximo.

Utilizando los datos del modelo, al comparar un núcleo liviano como el Calcio-40 frente a uno pesado como el Uranio-238, se observa que en el Calcio-40 el término de superficie penaliza el enlace con aproximadamente 5.35 MeV, dominando sobre su término de Coulomb de 2.08 MeV, mientras que en el Uranio-238 la situación se invierte y la repulsión de Coulomb penaliza fuertemente con 4.07 MeV frente a los 2.95 MeV de la superficie. Esta dinámica ocurre porque el efecto de superficie disminuye a medida que el núcleo crece geométricamente, pero simultáneamente la repulsión electrostática crece de forma cuadrática con la cantidad de protones. La competencia directa entre un término desestabilizador que decae y otro que aumenta rápidamente provoca que exista un punto de equilibrio óptimo en masas intermedias, generando el característico máximo en la curva de energía de enlace por nucleón.

2. El modelo predice su mayor enlace por nucleón en un $A$ distinto del ${}^{62}$Ni experimental. Calcule la diferencia de $E_B$ entre el máximo del modelo y el punto de su curva con $A=62$. Exprese también la diferencia como porcentaje de la magnitud de $E_B$ en ese punto. Discuta el tamaño de la diferencia y las limitaciones del modelo que podrían contribuir a ella. Distinga esta comparación entre puntos de la curva de una comparación directa con el núclido experimental: para esta última debe usar $Z=28$, $A=62$.

R/ La diferencia teórica de energía de enlace entre el máximo global predicho por el modelo para el $A=65$ y el valor evaluado en la curva teórica para el $A=62$ con $Z=28$ es de apenas $0.0053$ MeV, lo que representa una discrepancia minúscula de aproximadamente $0.061\%$ respecto a la magnitud en ese punto. Aunque numéricamente la diferencia es insignificante, el modelo falla en predecir empíricamente al Níquel-62 como el más estable porque la fórmula de la gota líquida es una aproximación macroscópica que asume al núcleo como un fluido continuo, omitiendo por completo los efectos de la estructura cuántica interna. Al ignorar el modelo cuántico de capas, la fórmula no puede contabilizar la enorme estabilidad adicional que otorgan los "números mágicos" de la naturaleza, como los 28 protones exactos que posee el isótopo del Níquel para cerrar una capa energética completa.


3. Para $A=56$, ¿qué $Z$ tiene el mayor enlace por nucleón en el modelo de cuatro términos? Compárelo con el hierro ($Z=26$): calcule la diferencia de energía **total**, indicando el orden de la resta, y diga cuál resulta más ligado.

   La fórmula completa incluye un quinto término, el de **apareamiento**: protones y neutrones tienden a formar pares. En la energía total, la corrección vale $-a_P A^{-1/2}$ si $Z$ y $N$ son ambos pares, $+a_P A^{-1/2}$ si ambos son impares, y cero si $A$ es impar, con $a_P\approx12$ MeV. Calcule la corrección para los dos núcleos comparados y determine si invierte su orden de enlace. Recuerde que esta corrección es de energía **total**: para sumarla a $E_B$ debe dividirla entre $A$.

   Este es un **cálculo adicional de comparación**: no incorpore el apareamiento a las funciones del modelo base ni regenere con él los archivos de D. Basta comparar los dos candidatos; no se pide repetir la búsqueda global con el modelo ampliado.

R/ Al evaluar el modelo base de cuatro términos para $A=56$, el núcleo con mayor enlace por nucleón resulta ser el Manganeso ($Z=25$), superando al Hierro ($Z=26$) al tener una energía total teóricamente más ligada por una diferencia de $1.198$ MeV. Sin embargo, al introducir la corrección de energía total del quinto término de apareamiento con $a_P \approx 12$ MeV, el Manganeso sufre una penalización de $+1.604$ MeV por tener un número impar tanto de protones como de neutrones, mientras que el Hierro recibe una bonificación de $-1.604$ MeV al ser un núcleo par-par. Al calcular nuevamente sus energías totales ajustadas, el orden de estabilidad se invierte por completo por una diferencia final de $2.010$ MeV a favor del Hierro, demostrando que la preferencia cuántica de los nucleones por formar pares es el factor decisivo que lo hace físicamente más estable.

4. Reutilizando las funciones de su módulo, calcule la energía liberada en la fisión simétrica idealizada ${}^{236}\mathrm{U}\to2\,{}^{118}\mathrm{Pd}$ ($Z=46$): $Q=M(92,236)-2M(46,118)$. Las masas se utilizan aquí como energías equivalentes en MeV. Como ambos lados tienen el mismo número total de protones, las masas electrónicas incluidas se cancelan en esta diferencia, dentro de la aproximación adoptada. Si $Q>0$, el canal es energéticamente favorable. Compárelo con la escala típica de unos 200 MeV por fisión del uranio y explique por qué este canal idealizado no reproduce todos los productos ni el reparto de energía de una fisión real.

R/ Calculando la energía liberada en la fisión simétrica idealizada del U-236 en dos fragmentos idénticos de Pd-118 mediante la diferencia de sus masas equivalentes teóricas, se obtiene un valor $Q$ de aproximadamente $180.3$ MeV. Este resultado teórico concuerda notablemente bien en orden de magnitud con la escala típica observable de unos 200 MeV por fisión del uranio. Sin embargo, este canal simétrico idealizado no reproduce una fisión real porque en la naturaleza la ruptura del núcleo es predominantemente asimétrica debido a la influencia de los niveles de energía de capas cuánticas, y además la resta directa de masas asume erróneamente que toda la energía se convierte en masa de los fragmentos, omitiendo la fracción térmica que se disipa de inmediato a través de la emisión de neutrones libres y rayos gamma prontos.   